# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all the record sets along with their `@id` fields, and then enumerate their fields and field `@id`s and names. This allows users to reference each entity by `@id`, ensuring precise handling and extraction.


In [ ]:
# List all available record sets and their associated fields
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Extract data from all record sets
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Loading records for the given record set by @id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print available columns in the first record set
if len(record_set_ids) > 0:
    print(f"Columns in first record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We will select a numeric field from one record set for demonstration. **Please ensure to adjust the `numeric_field_id` and `group_field_id` based on output from Section 2 above.**


In [ ]:
# Specify the record set and fields for EDA (replace with actual @id's from Section 2)
# For demonstration, we use the first record set and select numeric fields dynamically
selected_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_record_set_id] if selected_record_set_id else None
if df is not None:
    # Try to auto-detect a numeric field (float/int) for the demo
    numeric_field_id = None
    for col in df.columns:
        try:
            # Try to convert column to numeric
            df_numeric = pd.to_numeric(df[col], errors='coerce')
            if df_numeric.notnull().sum() > 0 and df_numeric.dtype != 'O':
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id:
        # Ensure column is float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].quantile(0.5)  # Use median as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to auto-detect a group-by field (categorical)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object' and df[col].nunique() < 10:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")
    else:
        print("No suitable numeric field found in the record set for EDA.")
else:
    print("No record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if df is not None and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field was selected, show boxplot by group
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


*In this notebook, we demonstrated how to load and explore a protein mass spectrometry dataset described by a Croissant schema using the `mlcroissant` library. By referencing all dataset entities via their `@id`, users can robustly interrogate each record set, field, and column. Through EDA, we showcased basic filtering, normalization, grouping, and visualization techniques. The approach here can be adapted for deeper analyses such as biomarker discovery or comparative protein studies as cited in the dataset metadata.*